In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/durgashyam9849@gmail.com/consolidated_pipeline/setup/utilities

In [0]:
dbutils.widgets.text('catalog','fmcg','Catalog')
dbutils.widgets.text('data_source', 'gross_price', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sportsbar-dem/{data_source}/*.csv'
print(base_path)

# Bronze

In [0]:
df_bronze = (spark.read.format("csv")
      .option("header", "true")
      .option("inferSchema", "true")
      .load(base_path)
      .withColumn("read_timestamp", F.current_timestamp())
      .select('*', '_metadata.file_name', '_metadata.file_size'))

In [0]:
df_bronze.printSchema()

In [0]:
display(df_bronze.limit(10))

In [0]:
display(df_bronze.select('month').distinct())

In [0]:
df_bronze.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

# Silver

In [0]:
def silver_trans(df):
  df_silver = df.withColumns({'month' :F.coalesce( F.try_to_date(F.col('month'), 'yyyy/MM/dd'),
            F.try_to_date(F.col('month'), 'dd/MM/yyyy'),
            F.try_to_date(F.col('month'), 'yyyy-MM-dd'),
            F.try_to_date(F.col('month'), 'dd-MM-yyyy')),
       
       'gross_price': F.when(F.col('gross_price').rlike(r'-?\d+(\.\d+)?$'), 
                      F.when(F.col('gross_price').cast('double') < 0, -1 * F.col('gross_price').cast('double'))
                      .otherwise(F.col('gross_price').cast('double'))).otherwise(0)})
  return df_silver

In [0]:
df_silver = silver_trans(df_bronze)
display(df_silver.limit(10))

# join the product_id from pricing_data and products table

In [0]:
df_products = spark.table("fmcg.silver.products")
df_joined = df_silver.join(df_products.select('product_id', 'product_code'), on = 'product_id', how = 'inner')
df_joined = df_joined.select('product_id', 'product_code', 'month', 'gross_price', 'read_timestamp', 'file_name', 'file_size')

df_joined.show(10)

In [0]:
df_joined.write\
.format("delta")\
.option("delta.enableChangeDataFeed", "true")\
.option("mergeSchema", "true").mode("overwrite")\
.saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

# Gold

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

In [0]:
df_gold = df_silver.select('product_code', 'month', 'gross_price')
display(df_gold.limit(10))

In [0]:
df_gold_updated = df_gold.withColumn('year', F.year('month'))
display(df_gold_updated)

In [0]:
%sql

with gross_p as (
    select product_code, 
    month,
    year(month) as year,
    gross_price,
    case when gross_price = 0 then 1 else 0 end as is_zero
    from fmcg.silver.gross_price
),

wind as (
    select *,
    row_number() over (partition by product_code, year 
    order by is_zero asc, month desc) AS rnk
    FROM gross_p
)

SELECT *
FROM wind
WHERE rnk = 1;

In [0]:
df_final = spark.sql("""
with gross_p as (
    select product_code, 
    month,
    year(month) as year,
    gross_price as price_inr,
    case when gross_price = 0 then 1 else 0 end as is_zero
    from fmcg.silver.gross_price
),

wind as (
    select *,
    row_number() over(partition by product_code, year 
    order by is_zero, month desc) AS rnk
    from gross_p
)

SELECT * from wind where rnk = 1 """)

In [0]:
display(df_final)

In [0]:
df_final = df_final.withColumn('year', F.col('year').cast('string'))
df_final = df_final.select('product_code', 'year', 'price_inr')
display(df_final)

In [0]:
df_gross_price = DeltaTable.forName(spark, "fmcg.gold.dim_gross_price")

df_gross_price.alias("target").merge(
    source = df_final.alias("source"),
    condition = "target.product_code = source.product_code")\
.whenMatchedUpdate(
    set = {"price_inr" : "source.price_inr",
        "year" : "source.year"})\
.whenNotMatchedInsert(
    values = {"product_code" : "source.product_code",
        "price_inr" : "source.price_inr",
        "year" : "source.year"}).execute()